# Implied Volatility Calculation Demo

This notebook demonstrates:
1. Black-Scholes option pricing
2. Implied volatility recovery via Newton-Raphson
3. 3-D surface plots of BS price and IV across strikes and vols


In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.stats import norm
from scipy.optimize import brentq

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Try to import from vol_surface_engine; fall back to local implementations
try:
    from vol_surface_engine.black_scholes import black_scholes_price, implied_vol_newton
    print('Loaded from vol_surface_engine')
except ImportError:
    print('vol_surface_engine not available — using local implementations')

    def black_scholes_price(
        S: float, K: float, T: float, r: float, sigma: float,
        q: float = 0.0, option_type: str = 'call'
    ) -> float:
        """Black-Scholes-Merton option price."""
        if T <= 0 or sigma <= 0:
            intrinsic = max(S * np.exp(-q * T) - K * np.exp(-r * T), 0)
            return intrinsic if option_type == 'call' else max(K * np.exp(-r * T) - S * np.exp(-q * T), 0)
        d1 = (np.log(S / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
        d2 = d1 - sigma * np.sqrt(T)
        if option_type == 'call':
            return S * np.exp(-q * T) * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
        else:
            return K * np.exp(-r * T) * norm.cdf(-d2) - S * np.exp(-q * T) * norm.cdf(-d1)

    def bs_vega(S: float, K: float, T: float, r: float, sigma: float, q: float = 0.0) -> float:
        """Black-Scholes vega (same for calls and puts)."""
        if T <= 0 or sigma <= 0:
            return 0.0
        d1 = (np.log(S / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
        return S * np.exp(-q * T) * norm.pdf(d1) * np.sqrt(T)

    def implied_vol_newton(
        market_price: float, S: float, K: float, T: float, r: float,
        q: float = 0.0, option_type: str = 'call',
        tol: float = 1e-8, max_iter: int = 100
    ) -> float:
        """Newton-Raphson implied volatility solver with Brent fallback."""
        # Initial guess: Brenner-Subrahmanyam approximation
        sigma = np.sqrt(2 * np.pi / T) * market_price / S
        sigma = np.clip(sigma, 0.01, 5.0)

        for _ in range(max_iter):
            price = black_scholes_price(S, K, T, r, sigma, q, option_type)
            vega = bs_vega(S, K, T, r, sigma, q)
            diff = price - market_price
            if abs(diff) < tol:
                return sigma
            if abs(vega) < 1e-12:
                break
            sigma -= diff / vega
            sigma = np.clip(sigma, 1e-4, 10.0)

        # Fallback: Brent's method
        try:
            def objective(s):
                return black_scholes_price(S, K, T, r, s, q, option_type) - market_price
            return brentq(objective, 1e-4, 10.0, xtol=tol)
        except ValueError:
            return float('nan')

print('Black-Scholes functions ready.')

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Black-Scholes pricing with standard SPX-like parameters
# ──────────────────────────────────────────────────────────────────────────────

S = 5000.0   # SPX spot price
K = 5000.0   # ATM strike
T = 30/252   # 30 calendar days to expiry
r = 0.05     # risk-free rate
q = 0.013    # dividend yield
sigma = 0.20 # assumed IV = 20%

call_price = black_scholes_price(S, K, T, r, sigma, q, 'call')
put_price  = black_scholes_price(S, K, T, r, sigma, q, 'put')

print(f'Black-Scholes Parameters:')
print(f'  S (spot)    = {S:,.0f}')
print(f'  K (strike)  = {K:,.0f}')
print(f'  T (years)   = {T:.4f}  ({T*252:.0f} calendar days)')
print(f'  r           = {r:.2%}')
print(f'  q           = {q:.2%}')
print(f'  σ (vol)     = {sigma:.2%}')
print()
print(f'Results:')
print(f'  Call price  = ${call_price:>10.4f}')
print(f'  Put  price  = ${put_price:>10.4f}')
print(f'  Put-call parity check: C - P = {call_price - put_price:.4f} ')
print(f'    Expected (S·e^(-qT) - K·e^(-rT)) = {S*np.exp(-q*T) - K*np.exp(-r*T):.4f}')

# Strike range for Greeks table
strikes_demo = np.array([4500, 4750, 5000, 5250, 5500])
rows = []
for K_i in strikes_demo:
    c = black_scholes_price(S, K_i, T, r, sigma, q, 'call')
    p = black_scholes_price(S, K_i, T, r, sigma, q, 'put')
    rows.append({'K': K_i, 'moneyness': K_i/S, 'call_price': c, 'put_price': p})

df_demo = pd.DataFrame(rows)
print(f'\nPrices across strikes (σ={sigma:.0%}, T={T*252:.0f}d):')
print(df_demo.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Implied volatility recovery via Newton-Raphson
# ──────────────────────────────────────────────────────────────────────────────

S = 5000.0; K = 5000.0; T = 30/252; r = 0.05; q = 0.013
true_vols = np.arange(0.10, 0.55, 0.05)

print('IV Recovery Accuracy Test (Newton-Raphson):')
print(f'{"True IV":>10}  {"Market Price":>14}  {"Recovered IV":>14}  {"Error (bps)":>12}')
print('-' * 56)

errors = []
for vol_true in true_vols:
    mkt_price = black_scholes_price(S, K, T, r, vol_true, q, 'call')
    vol_recovered = implied_vol_newton(mkt_price, S, K, T, r, q, 'call')
    error_bps = abs(vol_recovered - vol_true) * 10_000
    errors.append(error_bps)
    print(f'{vol_true:>10.2%}  {mkt_price:>14.4f}  {vol_recovered:>14.6f}  {error_bps:>12.4f}')

print('-' * 56)
print(f'Mean absolute error: {np.mean(errors):.6f} bps')
print(f'Max absolute error : {np.max(errors):.6f} bps')

# Test on OTM options
print('\nOTM Options IV Recovery (skewed surface):')
moneyness_range = np.array([0.85, 0.90, 0.95, 1.00, 1.05, 1.10, 1.15])
print(f'{"Moneyness":>12}  {"Strike":>8}  {"True IV":>9}  {"Recovered IV":>14}  {"Error (bps)":>12}')
print('-' * 63)
for m in moneyness_range:
    K_i = S * m
    # Skewed vol: higher for OTM puts (negative skew)
    k = np.log(m)
    vol_true = 0.20 - 0.30 * k + 0.50 * k**2
    vol_true = np.clip(vol_true, 0.05, 0.80)
    opt_type = 'put' if m <= 1.0 else 'call'
    mkt_price = black_scholes_price(S, K_i, T, r, vol_true, q, opt_type)
    vol_recovered = implied_vol_newton(mkt_price, S, K_i, T, r, q, opt_type)
    error_bps = abs(vol_recovered - vol_true) * 10_000
    print(f'{m:>12.2%}  {K_i:>8.0f}  {vol_true:>9.2%}  {vol_recovered:>14.6f}  {error_bps:>12.4f}')

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 3D surface: BS price and IV across strikes and implied vols
# ──────────────────────────────────────────────────────────────────────────────

S = 5000.0; T = 30/252; r = 0.05; q = 0.013
strikes = np.linspace(4000, 6000, 40)
vols    = np.linspace(0.05, 0.60, 40)
K_grid, vol_grid = np.meshgrid(strikes, vols)

price_surface = np.vectorize(
    lambda K, sigma: black_scholes_price(S, K, T, r, sigma, q, 'call')
)(K_grid, vol_grid)

fig_3d = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'surface'}, {'type': 'surface'}]],
    subplot_titles=['BS Call Price Surface', 'IV Smile (fixed params)']
)

# Left: price surface
fig_3d.add_trace(
    go.Surface(
        x=strikes,
        y=vols * 100,
        z=price_surface,
        colorscale='Viridis',
        colorbar=dict(title='Price ($)', x=0.45),
        showscale=True,
        name='BS Price',
    ),
    row=1, col=1
)

# Right: synthetic IV smile surface across strikes and expiries
expiries_days = np.array([7, 14, 21, 30, 45, 60, 90, 120, 180, 252])
strikes_smile = np.linspace(4000, 6000, 40)
iv_smile = np.zeros((len(expiries_days), len(strikes_smile)))

for i, dte in enumerate(expiries_days):
    Ti = dte / 252
    atm_iv = 0.18 + 0.02 * np.sqrt(Ti / (30/252))
    for j, Ki in enumerate(strikes_smile):
        k = np.log(Ki / S)
        rho_svi = -0.75; b_svi = 0.20; sigma_svi = 0.04; m_svi = 0.0
        a_svi = atm_iv**2 * Ti - b_svi * Ti * (rho_svi * (0 - m_svi) + np.sqrt((0 - m_svi)**2 + sigma_svi**2))
        w = a_svi + b_svi * Ti * (rho_svi * (k - m_svi) + np.sqrt((k - m_svi)**2 + sigma_svi**2))
        iv_smile[i, j] = np.sqrt(max(w, 1e-8) / Ti) * 100  # in %

fig_3d.add_trace(
    go.Surface(
        x=strikes_smile,
        y=expiries_days,
        z=iv_smile,
        colorscale='RdBu_r',
        colorbar=dict(title='IV (%)', x=1.02),
        showscale=True,
        name='IV Surface',
    ),
    row=1, col=2
)

fig_3d.update_layout(
    title='Black-Scholes Price Surface and SVI Implied Volatility Surface',
    scene=dict(
        xaxis_title='Strike (K)',
        yaxis_title='Implied Vol (%)',
        zaxis_title='Call Price ($)',
    ),
    scene2=dict(
        xaxis_title='Strike (K)',
        yaxis_title='DTE',
        zaxis_title='Implied Vol (%)',
    ),
    width=1200,
    height=600,
    template='plotly_white',
)

from plotly.subplots import make_subplots
fig_3d.show()
print('3D surfaces rendered.')